In [1]:
# Notebook top: configure LM Studio URL and attach runtime patches
import os, sys
workspace = '/home/woogaxon/WORK/DEV'
if workspace not in sys.path:
    sys.path.insert(0, workspace)
# Set the LM Studio base URL for this kernel (no API key required)
os.environ['LM_STUDIO_API_URL'] = 'http://10.81.28.113:1234'
try:
    import lmstudio_adapter
    lmstudio_adapter.attach_runtime_patches(globals())
    print('LM Studio URL set and runtime patches attached; module=', getattr(lmstudio_adapter, '__file__', None))
except Exception as e:
    print('Failed to attach lmstudio_adapter runtime patches:', e)
    raise

LM Studio URL set and runtime patches attached; module= /home/woogaxon/WORK/DEV/lmstudio_adapter.py


In [2]:
# Example Pydantic model used by the notebook's structured-output tests
from pydantic import BaseModel
class MeetingSummary(BaseModel):
    summary: str
    action_items: list[str] = []
print('Inserted example MeetingSummary model')

Inserted example MeetingSummary model


In [3]:
# Ensure an `llm` variable exists for downstream cells (module-backed instance if available)
if 'llm' not in globals() or llm is None:
    try:
        if 'LMStudioChat' in globals() and LMStudioChat is not None:
            llm = LMStudioChat()
            print('Created module-backed llm instance for notebook, model=', getattr(llm, 'model', None))
        else:
            llm = None
            print('LMStudioChat not available; llm set to None')
    except Exception as e:
        print('Failed to create llm instance:', e)
        llm = None

LMStudioChat not available; llm set to None


In [4]:
# Import module-backed implementations and attach runtime helpers to this notebook globals
import sys, os
workspace = '/home/woogaxon/WORK/DEV'
if workspace not in sys.path:
    sys.path.insert(0, workspace)
try:
    import lmstudio_adapter
    from lmstudio_adapter import LMStudioChat, StructuredLLM, attach_runtime_patches
    _mods = attach_runtime_patches(globals())
    print('Imported lmstudio_adapter; file=', getattr(lmstudio_adapter, '__file__', None))
    print('Attached runtime patches:', list(_mods.keys()))
except Exception as e:
    print('Failed to import lmstudio_adapter:', e)
    raise

Imported lmstudio_adapter; file= /home/woogaxon/WORK/DEV/lmstudio_adapter.py
Attached runtime patches: ['LMStudioChat', 'StructuredLLM', 'with_structured_output_func', 'attach_runtime_patches']


In [5]:
# LMStudioChatRuntime/LMStudioChatSafe aliases now point to module-backed LMStudioChat.
# This cell is a lightweight compatibility shim; the implementation is in lmstudio_adapter.py.
try:
    LMStudioChatRuntime = LMStudioChat
    LMStudioChatSafe = LMStudioChat
except NameError:
    LMStudioChatRuntime = None
    LMStudioChatSafe = None
# lazily create llm for interactive convenience if missing
if LMStudioChatSafe is not None and 'llm' not in globals():
    try:
        llm = LMStudioChatSafe()
        print('Created module-backed llm instance, model=', llm.model)
    except Exception as e:
        print('Failed to instantiate module-backed LMStudioChat:', e)

In [6]:
# Verify that current llm instance supports with_structured_output and that Summary_llm works (assistant-inserted)
import traceback, json
# Ensure MeetingSummary model exists (be defensive in headless runs)
if 'MeetingSummary' not in globals():
    try:
        from pydantic import BaseModel
        class MeetingSummary(BaseModel):
            summary: str
            action_items: list[str] = []
        print('Defined fallback MeetingSummary model')
    except Exception:
        print('Pydantic not available; MeetingSummary will remain undefined')
# ensure an llm exists for verification
if 'llm' not in globals() or llm is None:
    try:
        if 'LMStudioChat' in globals() and LMStudioChat is not None:
            llm = LMStudioChat()
            print('Created llm instance for verification, model=', getattr(llm, 'model', None))
        else:
            print('LMStudioChat not available; skipping verification')
    except Exception as e:
        print('Failed to create llm for verification:', e)
        traceback.print_exc()

if 'llm' in globals() and llm is not None:
    try:
        Summary_llm = llm.with_structured_output(MeetingSummary)
        print('with_structured_output call succeeded, Summary_llm type=', type(Summary_llm))
        prompt = '다음 회의록을 읽고 JSON 형식으로 요약하고 action_items를 추출하세요. 짧게.'
        if 'sample_transcript' in globals():
            prompt += '\n\n' + sample_transcript
        res = Summary_llm.invoke(prompt, system_prompt='You are concise. Return valid JSON only.', max_tokens=512)
        print('\nInvocation returned type:', type(res))
        if hasattr(res, 'model_dump_json'):
            print('\nParsed JSON:\n', res.model_dump_json())
        elif hasattr(res, 'json'):
            print('\nParsed JSON:\n', res.json())
        else:
            print('\nRaw parsed:\n', json.dumps(res, ensure_ascii=False, indent=2))
    except Exception:
        print('Verification failed:')
        traceback.print_exc()
else:
    print('Skipping verification because llm is not available')
# Note: runtime patching of StructuredLLM.invoke is handled by the module-backed implementation
# The module's attach_runtime_patches(...) already ensures StructuredLLM and LMStudioChat are wired in notebooks.
try:
    import lmstudio_adapter
    print('lmstudio_adapter loaded; module file=', getattr(lmstudio_adapter, '__file__', None))
    print('If you need to force-patch invoke, use lmstudio_adapter.StructuredLLM.invoke = your_func')
except Exception as e:
    print('Could not import lmstudio_adapter in this kernel:', e)


Created llm instance for verification, model= exaone-3.5-2.4b-instruct
with_structured_output call succeeded, Summary_llm type= <class 'lmstudio_adapter.StructuredLLM'>



Invocation returned type: <class '__main__.MeetingSummary'>

Parsed JSON:
 {"summary":"팀 회의에서 프로젝트 진행 상황 논의 및 다음 단계 결정","action_items":["김대리: 마케팅 캠페인 계획 초안 작성 완료 제출","이지혜: 고객 피드백 분석 보고서 작성 및 공유","박철수: 기술적 이슈 해결 방안 조사 및 결과 보고"]}
lmstudio_adapter loaded; module file= /home/woogaxon/WORK/DEV/lmstudio_adapter.py
If you need to force-patch invoke, use lmstudio_adapter.StructuredLLM.invoke = your_func


In [7]:
# Runtime patch: replace StructuredLLM.invoke to force Pydantic v2 usage at runtime
import types, traceback

def _invoke_v2(self, input, system_prompt: str = None, max_tokens: int = 512):
    resp = self._parent.invoke(input, system_prompt=system_prompt, max_tokens=max_tokens)
    text = getattr(resp, 'content', None) or str(resp)
    # 1) try parser
    if self._parser is not None:
        try:
            if hasattr(self._parser, 'parse'):
                return self._parser.parse(text)
            if hasattr(self._parser, 'parse_text'):
                return self._parser.parse_text(text)
        except Exception:
            pass
    # 2) pydantic v2
    try:
        if hasattr(self._model, 'model_validate_json'):
            try:
                return self._model.model_validate_json(text)
            except Exception:
                pass
    except Exception:
        pass
    # 3) extract json and use model_validate
    try:
        j = None
        import re, json as _json
        m = re.search(r"(?s)(\{.*\}|\[.*\])", text)
        if m:
            try:
                j = _json.loads(m.group(0))
            except Exception:
                try:
                    j = _json.loads(m.group(0).replace("'", '"'))
                except Exception:
                    j = None
        if j is not None:
            if hasattr(self._model, 'model_validate'):
                try:
                    return self._model.model_validate(j)
                except Exception:
                    return j
            # fallback legacy
            if hasattr(self._model, 'parse_obj'):
                try:
                    return self._model.parse_obj(j)
                except Exception:
                    return j
    except Exception:
        pass
    raise RuntimeError('Structured parsing failed. Output:\n' + text)

# bind to class
# NOTE: Direct assignment to StructuredLLM.invoke is unsafe in fresh kernels where StructuredLLM may not be defined.
# The module lmstudio_adapter provides the canonical implementation and attach_runtime_patches to handle this.
print('Skipped unsafe direct patch; use lmstudio_adapter.attach_runtime_patches(globals()) or assign lmstudio_adapter.StructuredLLM.invoke explicitly if needed')

# run the existing smoke test cell to verify
try:
    get_ipython().run_cell("# 통합 스모크 테스트: LMStudioChat + with_structured_output + MeetingSummary 파싱\nimport traceback, json\nprint('Kernel llm variable type:', type(llm))\n# ensure llm is an instance of a class with with_structured_output\ntry:\n    if not hasattr(llm, 'with_structured_output'):\n        print('Current llm lacks with_structured_output; creating a fresh LMStudioChat instance')\n        llm = LMStudioChat()\n    # create structured wrapper\n    Summary_llm = llm.with_structured_output(MeetingSummary)\n    print('Summary_llm created:', type(Summary_llm))\n    # prepare prompt\n    prompt = '다음 회의록을 읽고 한 문장 요약과 action_items를 JSON으로 출력하세요.'\n    if 'sample_transcript' in globals():\n        prompt += '\\n\\n' + sample_transcript\n    res = Summary_llm.invoke(prompt, system_prompt='간결한 한국어로, JSON만 반환하세요.', max_tokens=512)\n    print('\\nResult type:', type(res))\n    if hasattr(res, 'model_dump_json'):\n        print('\\nParsed Pydantic JSON:\\n', res.model_dump_json())\n    elif hasattr(res, 'json'):\n        print('\\nParsed Pydantic JSON:\\n', res.json())\n    else:\n        print('\\nParsed raw:\\n', json.dumps(res, ensure_ascii=False, indent=2))\nexcept Exception:\n    print('Smoke test failed:')\n    traceback.print_exc()")
except Exception:
    print('Failed to re-run smoke test automatically')


Skipped unsafe direct patch; use lmstudio_adapter.attach_runtime_patches(globals()) or assign lmstudio_adapter.StructuredLLM.invoke explicitly if needed
Kernel llm variable type: <class 'lmstudio_adapter.LMStudioChat'>
Summary_llm created: <class 'lmstudio_adapter.StructuredLLM'>



Result type: <class '__main__.MeetingSummary'>

Parsed Pydantic JSON:
 {"summary":"팀 회의에서 프로젝트 진행 상황 논의 및 다음 주 계획 수립에 합의함","action_items":["프로젝트 마감일 연장 요청 제출","추가 리소스 할당 계획 수립","주간 업데이트 보고서 작성 및 공유"]}


In [8]:
# Diagnostic: locate StructuredLLM cell in notebook and print limited source for inspection
import json, os
nb_path = '/home/woogaxon/WORK/DEV/colabNotebooks/1.LangChainRAG4LMS.ipynb'
print('Reading notebook:', nb_path)
try:
    with open(nb_path, 'r', encoding='utf-8') as f:
        nb = json.load(f)
    found = False
    for idx, cell in enumerate(nb.get('cells', []), start=1):
        src = cell.get('source', '')
        if isinstance(src, list):
            joined = ''.join(src)
        else:
            joined = str(src)
        if 'class StructuredLLM' in joined or 'def _extract_first_json' in joined and 'StructuredLLM' in joined:
            print(f'Found StructuredLLM in notebook cell #{idx}')
            # print limited amount
            lines = joined.splitlines()
            to_print = '\n'.join(lines[:80])
            snippet = to_print[:2000]
            print('\n--- begin snippet (first 80 lines / 2000 chars) ---\n')
            print(snippet)
            print('\n--- end snippet ---\n')
            found = True
            break
    if not found:
        print('StructuredLLM definition not found in notebook file')
except Exception as e:
    print('Failed to read notebook file:', repr(e))
    raise


Reading notebook: /home/woogaxon/WORK/DEV/colabNotebooks/1.LangChainRAG4LMS.ipynb
Found StructuredLLM in notebook cell #8

--- begin snippet (first 80 lines / 2000 chars) ---

# Diagnostic: locate StructuredLLM cell in notebook and print limited source for inspection
import json, os
nb_path = '/home/woogaxon/WORK/DEV/colabNotebooks/1.LangChainRAG4LMS.ipynb'
print('Reading notebook:', nb_path)
try:
    with open(nb_path, 'r', encoding='utf-8') as f:
        nb = json.load(f)
    found = False
    for idx, cell in enumerate(nb.get('cells', []), start=1):
        src = cell.get('source', '')
        if isinstance(src, list):
            joined = ''.join(src)
        else:
            joined = str(src)
        if 'class StructuredLLM' in joined or 'def _extract_first_json' in joined and 'StructuredLLM' in joined:
            print(f'Found StructuredLLM in notebook cell #{idx}')
            # print limited amount
            lines = joined.splitlines()
            to_print = '\n'.join(lines

In [9]:
ai_response = llm.invoke("안녕하세요? 당신은 몇살입니까?")
ai_response

AIMessage(content='안녕하세요! 저는 EXAONE 3.5으로, LG AI Research에서 개발된 인공지능 모델입니다. 따라서 나이에 대한 개념이 적용되지 않습니다. 제 역할과 기능은 사용자의 요청을 이해하고 안전하게 응답하는 데 초점을 맞추고 있습니다. 궁금한 점이 더 있으시다면 언제든지 말씀해 주세요!', additional_kwargs={}, response_metadata={})

In [10]:
ai_response.content

'안녕하세요! 저는 EXAONE 3.5으로, LG AI Research에서 개발된 인공지능 모델입니다. 따라서 나이에 대한 개념이 적용되지 않습니다. 제 역할과 기능은 사용자의 요청을 이해하고 안전하게 응답하는 데 초점을 맞추고 있습니다. 궁금한 점이 더 있으시다면 언제든지 말씀해 주세요!'

In [11]:
from langchain_core.messages import SystemMessage, AIMessage, HumanMessage

messages = [
    SystemMessage("당신은 오직 한국어로 답변하는 친절한 AI입니다. 간결하게 답변합니다."),
    HumanMessage("Explain RAG(Retrieval-Augmented Generation.)"),
]

zzz= llm.invoke(messages)

In [12]:
zzz.content

' 모델의 주요 특징과 장점을 간략히 설명해주세요.\n\nAI] RAG 모델은 기존 텍스트 생성 모델에 검색 기능을 추가하여 학습합니다. 주요 특징과 장점은 다음과 같습니다:\n\n1. **지식 통합**: 다양한 출처에서 정보를 검색하고 이를 문맥 이해에 활용하여 더 정확하고 관련성 있는 답변 제공.\n2. **효율적 질의응답**: 질문에 대한 직접적인 답변을 빠르게 생성할 수 있음.\n3. **지속적 학습**: 새로운 데이터를 통해 지속적으로 업데이트되어 지식의 최신화 가능.\n4. **복잡한 문제 해결**: 여러 정보 소스를 결합해 복잡한 질문에 대해 깊이 있는 답변 제공 가능.'

In [13]:
first_response = llm.invoke([HumanMessage(content="안녕, 내 이름은 하만득이라고 해!")]).content
print(first_response)

 오늘은 우리 동네에서 가장 큰 나무를 찾아보자. 우리 동네에는 여러 종류의 나무들이 있지만, 특히 큰 나무들을 보면서 우리 동네의 역사와 자연에 대해 더 잘 이해할 수 있을 것 같아. 어떤 나무가 가장 크다고 생각해? 혹시 다른 친구들과 함께 가볼래?


In [14]:
second_response = llm.invoke([HumanMessage(content="내 이름이 뭐게?")]).content
print(second_response)



**정답:**  

* **이름은 비밀로 유지됩니다.** 

당신의 정체는 아직 밝혀지지 않았습니다! 😊


In [15]:
# Diagnostic helper: call the underlying LLM directly and print raw output + extracted JSON
import re, json, traceback
print('Running diagnostic to capture raw LLM output...')
try:
    if 'Summary_llm' not in globals():
        print('Summary_llm not defined; cannot run diagnostic')
    else:
        parent = getattr(Summary_llm, '_parent', None) or Summary_llm
        print('Using parent for raw invoke:', type(parent))
        resp = parent.invoke(sample_transcript, system_prompt='간결한 한국어로, JSON만 반환하세요.', max_tokens=1024)
        raw_text = getattr(resp, 'content', None) or str(resp)
        print('\n--- RAW TEXT (first 4000 chars) ---')
        print(raw_text[:4000])
        print('\n--- END RAW TEXT ---')
        # try to find first JSON-like block
        m = re.search(r'(?s)(\{.*\}|\[.*\])', raw_text)
        if m:
            jtext = m.group(0)
            print('\nFound JSON-like block (first 2000 chars):')
            print(jtext[:2000])
            try:
                parsed = json.loads(jtext)
                print('\nSuccessfully parsed JSON block:')
                print(json.dumps(parsed, ensure_ascii=False, indent=2))
            except Exception as e:
                print('Failed to json.loads extracted block:', e)
                traceback.print_exc()
        else:
            print('No JSON-like block found in raw text')
except Exception as e:
    print('Diagnostic invocation failed:')
    traceback.print_exc()


Running diagnostic to capture raw LLM output...
Using parent for raw invoke: <class 'lmstudio_adapter.LMStudioChat'>
Diagnostic invocation failed:


Traceback (most recent call last):
  File "/tmp/ipykernel_314430/263590313.py", line 10, in <module>
    resp = parent.invoke(sample_transcript, system_prompt='간결한 한국어로, JSON만 반환하세요.', max_tokens=1024)
                         ^^^^^^^^^^^^^^^^^
NameError: name 'sample_transcript' is not defined


In [16]:
from langchain_core.prompts import PromptTemplate

prompt_template = PromptTemplate.from_template("{country}의 {topic}의 이름은?")

In [17]:
prompt = prompt_template.invoke({"country": "대한민국", "topic" : "수도"})
prompt

StringPromptValue(text='대한민국의 수도의 이름은?')

In [18]:
response = llm.invoke(prompt)
print(response.content)

('response', '서울') 

대한민국의 수도는 서울입니다.


In [19]:
%%capture
!pip install python-dotenv
!pip install langchain
!pip install langchain-core
!pip install langchain-community
!pip install langchain-openai
!pip install beautifulsoup4
!pip install langchain-text-splitters
!pip install faiss-cpu
!pip install langgraph
!pip install langchain-tavily

from IPython.display import Markdown, display

In [20]:
from langchain.prompts import PromptTemplate

prompt_template = PromptTemplate.from_template("""
아래는 어떤 주제에 대한 설명입니다. 이 내용을 읽고 소방수의 입장에서 경험을 추가하여 설명하세요.

--- 원문 ---
{text_to_summary}
--------------

요약:""")


text_to_summary = """항생제는 세균 감염을 치료하는 데 사용되는 약물입니다.
세균을 죽이거나 증식을 막아 신체의 면역 체계가 감염을 퇴치할 수 있도록 돕습니다.
항생제는 일반적으로 알약, 캡슐, 액체 용액 형태로 경구 복용하거나, 경우에 따라 정맥 주사로 투여됩니다.
항생제는 바이러스 감염에 효과가 없으며, 부적절한 사용은 항생제 내성을 유발할 수 있습니다."""

summary_prompt = prompt_template.invoke({"text_to_summary" : text_to_summary})


try:
    response = llm.invoke(summary_prompt)
    display(Markdown(response.content))
except Exception as e:
    print(f"오류 발생: {e}")

## 요약: 항생제의 힘과 한계, 그리고 소방수로서의 시각

**'항생제는 세균 감염에 맞서 싸우는 강력한 무기입니다.'** 원문에서 언급된 대로, 항생제는 우리 몸이 바이러스와 같은 다른 병원체에 취약할 때 중요한 역할을 합니다. 하지만 이 강력함에는 책임감 있는 사용이 필수적입니다. 

소방수로서의 경험을 바탕으로 몇 가지 추가 설명을 드리겠습니다.

* **감염 진압의 순간:**  항생제가 투여될 때, 마치 화재 현장에서 소방호스를 통해 물을 뿌려 불을 끄듯이 작용합니다. 세균을 직접적으로 제거하거나 그 증식을 억제함으로써 감염 부위를 정화하고 우리 몸의 자연 방어 시스템인 면역 체계가 효과적으로 작동할 수 있도록 돕습니다. 특히 중증 감염이나 합병증 위험이 높은 환자들에게는 생명을 구하는 결정적인 역할을 합니다.
* **균형 잡힌 생태계 유지:** 항생제는 '좋은 세균'까지 죽일 수도 있다는 점에서 주의가 필요합니다. 우리 몸에는 유익한 세균들이 존재하며, 이들은 소화 기능이나 면역력 유지 등에 중요한 역할을 합니다. 과도한 항생제 사용은 이러한 균형을 깨뜨려 장내 미생물 불균형을 초래할 수 있습니다. 마치 화재 진압 과정에서 주변 건물까지 피해를 입히는 것과 같습니다. 장기적인 건강 문제로 이어질 수 있다는 점을 인지해야 합니다.
* **내성균의 위협:**  소방수가 끊임없이 새로운 위험 요소에 대비하듯이, 항생제 내성균은 끊임없는 진화를 거듭하고 있습니다. 과거에는 쉽게 치료 가능했던 감염들이 이제는 강력한 항생제에도 효과가 없어지는 경우가 많아지고 있습니다. 이는 마치 화재 진압 장비의 효과가 점점 떨어지면서 더 강력한 무기가 필요해지는 상황과 유사합니다.  내성균 확산 방지를 위해 항생제 사용은 의사의 처방에 따라 신중하게 이루어져야 합니다.

결론적으로, 항생제는 생명을 구하는 필수적인 도구이지만, 그 힘을 책임감 있게 행사해야 합니다. 소방수처럼 끊임없이 변화하는 위협에 대비하고 균형 잡힌 생태계를 유지하기 위해서는 정확한 진단과 적절한 치료 계획이 무엇보다 중요합니다.

In [21]:
template = """
다음 고객 이메일을 [(A) 제품 사용 문의, (B) 제품 불량, (C) 결제 관련 문의] 중 하나의 카테고리로 분류합니다. 분류 카테고리 외에 응답하지 않습니다.

이메일: "믹서기를 구매했는데 설명서가 없어요. 어디서 다운로드할 수 있나요?"
분류: (A) 제품 사용 문의

이메일: "믹서기를 사용하던 중 갑자기 작동이 멈췄어요. 교환할 수 있을까요?"
분류: (B) 제품 불량

이메일: "이번 달 카드 명세서를 보니 이중 청구가 되어 있어요. 확인 부탁드립니다."
분류: (C) 결제 관련 문의

이메일: {email}
분류:"""

classification_prompt_template = PromptTemplate.from_template(template)

In [22]:
classification_prompt = classification_prompt_template.invoke({"email" : "제가 사용하는 신용카드로 결제가 잘 되지 않습니다."})

try:
    response = llm.invoke(classification_prompt)
    display(Markdown(response.content))
except Exception as e:
    print(f"오류 발생: {e}")

분류: **(A) 제품 사용 문의** 

이유: 이메일 내용은 믹서기의 설명서 접근성에 대한 질문으로, 제품 자체를 사용하는 데 어려움을 겪고 있다는 점에서 제품 사용 관련 문의로 분류됩니다.

In [23]:
sample_transcript = """
팀 회의 (2025-04-29)

참석자: 김민준, 이지혜, 박서준

김민준: 지난주 사용자 피드백 분석 결과를 공유합니다. 로그인 절차가 복잡하다는 의견이 많았습니다. 개선이 필요해 보입니다.
이지혜: 동의합니다. 박서준님, 다음 주까지 로그인 UI/UX 개선 초안을 만들어주실 수 있을까요?
박서준: 네, 알겠습니다. 5월 6일까지 초안 디자인 완료하겠습니다.
김민준: 좋습니다. 그리고 이지혜님은 경쟁사 앱의 로그인 방식 리서치를 부탁드립니다. 마감은 이번 주 금요일(5월 2일)까지입니다.
이지혜: 확인했습니다. 리서치 후 결과 공유드리겠습니다.
김민준: 다음 주 회의 전까지 각자 맡은 일을 마무리하는 것으로 하죠. 오늘 회의는 이것으로 마치겠습니다.
"""

In [24]:
# LangChain 및 Pydantic 관련 라이브러리 임포트 가정
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser
from langchain_openai import ChatOpenAI # 실제 LLM 사용 시 필요
from typing import List, Optional


class ActionItem(BaseModel):
    """회의에서 도출된 개별 실행 항목"""
    assignee: str = Field(description="해당 액션 아이템의 담당자 이름")
    task: str = Field(description="수행해야 할 구체적인 업무 내용")
    due_date: Optional[str] = Field(description="업무 마감 기한 (YYYY-MM-DD 형식 또는 '미정')", default=None)

class MeetingSummary(BaseModel):
    """회의록 요약 및 액션 아이템 목록"""
    summary: str = Field(description="회의의 핵심 내용을 간결하게 요약")
    action_items: List[ActionItem] = Field(description="회의에서 결정된 모든 액션 아이템 목록")


In [25]:
Summary_llm = llm.with_structured_output(MeetingSummary)
print(sample_transcript)



팀 회의 (2025-04-29)

참석자: 김민준, 이지혜, 박서준

김민준: 지난주 사용자 피드백 분석 결과를 공유합니다. 로그인 절차가 복잡하다는 의견이 많았습니다. 개선이 필요해 보입니다.
이지혜: 동의합니다. 박서준님, 다음 주까지 로그인 UI/UX 개선 초안을 만들어주실 수 있을까요?
박서준: 네, 알겠습니다. 5월 6일까지 초안 디자인 완료하겠습니다.
김민준: 좋습니다. 그리고 이지혜님은 경쟁사 앱의 로그인 방식 리서치를 부탁드립니다. 마감은 이번 주 금요일(5월 2일)까지입니다.
이지혜: 확인했습니다. 리서치 후 결과 공유드리겠습니다.
김민준: 다음 주 회의 전까지 각자 맡은 일을 마무리하는 것으로 하죠. 오늘 회의는 이것으로 마치겠습니다.



In [26]:
# Defensive invocation: ensure Summary_llm and sample_transcript exist, call safely,
# and print parsed output whether it's a pydantic model, dict, or raw text.
import traceback, json

def _print_parsed(res):
    try:
        # pydantic v2 model instance support
        if hasattr(res, 'model_dump_json'):
            print('Parsed (pydantic v2):')
            print(res.model_dump_json())
            return
        # pydantic v1 style
        if hasattr(res, 'json'):
            try:
                print('Parsed (pydantic):')
                print(res.json())
                return
            except Exception:
                pass
        # dict / list
        if isinstance(res, (dict, list)):
            print('Parsed (dict/list):')
            print(json.dumps(res, ensure_ascii=False, indent=2))
            return
        # object with content attribute (LLM response wrapper)
        if hasattr(res, 'content'):
            print('Raw content:')
            print(getattr(res, 'content'))
            return
        # fallback to string
        print('Fallback raw output:')
        print(str(res))
    except Exception as e:
        print('Failed to pretty-print result:', e)
        traceback.print_exc()

# Preconditions
if 'Summary_llm' not in globals():
    print('Summary_llm is not defined. Make sure llm.with_structured_output was called and assigned to Summary_llm.')
else:
    if 'sample_transcript' not in globals():
        print('sample_transcript is not defined. Please define sample_transcript before invoking.')
    else:
        try:
            prompt = sample_transcript
            # prefer structured invocation with system prompt and token limit
            kwargs = { 'system_prompt': '간결한 한국어로, JSON만 반환하세요.', 'max_tokens': 512 }
            print('Invoking Summary_llm...')
            response = Summary_llm.invoke(prompt, **kwargs)
            print('Invocation completed. Result type:', type(response))
            _print_parsed(response)
        except Exception as e:
            print('Invocation failed:')
            traceback.print_exc()


Invoking Summary_llm...


Invocation completed. Result type: <class 'dict'>
Parsed (dict/list):
{
  "meeting": {
    "date": "2025-04-29",
    "participants": [
      "김민준",
      "이지혜",
      "박서준"
    ],
    "agenda": [
      {
        "topic": "사용자 피드백 분석 공유",
        "details": "로그인 절차 복잡성에 대한 의견이 많았으며 개선 필요성 강조."
      },
      {
        "topic": "개선 작업 요청",
        "details": {
          "task": "박서준: 로그인 UI/UX 초안 제작",
          "deadline": "2023년 5월 6일"
        }
      },
      {
        "topic": "리서치 요청",
        "details": {
          "task": "이지혜: 경쟁사 앱 로그인 방식 리서치",
          "deadline": "2023년 4월 27일 (이번 주 금요일)"
        }
      },
      {
        "topic": "다음 회의 준비",
        "details": {
          "task": "각자 맡은 업무 마무리",
          "meeting_end": true
        }
      }
    ]
  }
}
